In [0]:
-- status_bronze_demo
CREATE OR REFRESH STREAMING TABLE status_bronze_streaming 
    COMMENT 'Ingest Json status from cloud storage'
    TBLPROPERTIES (
        "quality" = "bronze",
        "pipelines.reset.allowed" = false  --prevent full table refresh from bronze table
    )
AS
SELECT *,
        current_timestamp() AS processing_time,
        _metadata.file_name as source_file
FROM STREAM read_files(
    '${source}/status',
    format => 'json'
);

In [0]:
-----------Silver - streaming table ---------------
CREATE OR REFRESH STREAMING TABLE status_silver_streaming 
(
        CONSTRAINT valid_timestamp EXPECT (order_status_timestamp > '2021-12-25') ON VIOLATION DROP ROW,
        CONSTRAINT valid_order_status EXPECT(order_status IN ('on the way','return canceled','delivered','return processed','placed','preparing','canceled'))
)
    COMMENT 'Order with status and timestamp'
    TBLPROPERTIES ("quality" = "silver")
AS
SELECT order_id,
        order_status,
        timestamp(status_timestamp) AS order_status_timestamp
FROM STREAM status_bronze_streaming;


In [0]:
CREATE OR REFRESH MATERIALIZED VIEW full_order_info_gold 
    COMMENT 'Joing 2 streaming table orders and status'
    TBLPROPERTIES ("quality" = "gold")
AS
SELECT o.order_id,
        o.order_timestamp,
        s.order_status,
        s.order_status_timestamp
FROM status_silver_streaming s
INNER JOIN order_silver_streaming o ON s.order_id = o.order_id;

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW cancelled_orders_gold 
    COMMENT 'Canceled orders'
    TBLPROPERTIES ("quality" = "gold")
AS
SELECT order_id,
        order_timestamp,
        order_status,
        order_status_timestamp,
        datediff(DAY, order_timestamp, order_status_timestamp) as days_to_canceled
FROM full_order_info_gold
WHERE order_status = 'canceled';

In [0]:
CREATE OR REFRESH MATERIALIZED VIEW delivered_orders_gold 
    COMMENT 'Delivered orders'
    TBLPROPERTIES ("quality" = "gold")
AS
SELECT order_id,
        order_timestamp,
        order_status,
        order_status_timestamp,
        datediff(DAY, order_timestamp, order_status_timestamp) as days_to_canceled
FROM full_order_info_gold
WHERE order_status = 'delivered';